# Construction d'un modèle de prédiction du cancer du sein à partir de biomarqueurs
**Dataset** : Breast Cancer Coimbra - UCI Machine Learning Repository  

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
RANDOM_STATE = 42
print('Imports done')


## A - Analyse Exploratoire des Données (EDA)

### A.1 - Chargement et aperçu général

In [ ]:
# Charger les données
df = pd.read_csv('dataR2.csv')

print(f'Dimensions : {df.shape[0]} observations × {df.shape[1]} variables')
print(f'\nColonnes : {df.columns.tolist()}')
print('\n Aperçu (5 premières lignes) ')
df.head()

In [ ]:
# Informations sur les types et valeurs manquantes
print(' Types et valeurs manquantes ')
info_df = pd.DataFrame({
    'Type': df.dtypes,
    'Non-nuls': df.notnull().sum(),
    'Nuls': df.isnull().sum(),
    '% Nuls': (df.isnull().sum() / len(df) * 100).round(2)
})
print(info_df)
print(f'\n Aucune valeur manquante.' if df.isnull().sum().sum() == 0 else ' Valeurs manquantes détectées.')

> **Note** : Conformément à la documentation UCI, le dataset ne contient aucune valeur manquante. Toutes les variables sont numériques continues, à l'exception de `Classification` (variable cible binaire : 1 = contrôle sain, 2 = patiente atteinte d'un cancer du sein).

### A.2 - Statistiques descriptives

In [ ]:
# Séparer features et cible
features = [c for c in df.columns if c != 'Classification']
target = 'Classification'

# Statistiques descriptives détaillées
data = df[features]
desc = pd.DataFrame({
    'Moyenne':       data.mean(),
    'Écart-type':    data.std(),
    'Min':           data.min(),
    'Q1 (25%)':      data.quantile(0.25),
    'Médiane (50%)': data.quantile(0.50),
    'Q3 (75%)':      data.quantile(0.75),
    'Max':           data.max(),
    'Asymétrie':     data.apply(stats.skew),
    'Kurtosis':      data.apply(stats.kurtosis),
}).round(3)
print(' Statistiques descriptives ')
desc

In [ ]:
# Statistiques par classe
print(' Statistiques par classe (1=Contrôle, 2=Cancer) ')
df.groupby(target)[features].mean().round(3)

### A.3 - Vérification du déséquilibre des classes

In [ ]:
class_counts = df[target].value_counts().sort_index()
class_labels = {1: 'Contrôle sain (1)', 2: 'Cancer du sein (2)'}
class_counts.index = [class_labels[i] for i in class_counts.index]

print(' Distribution des classes ')
for label, count in class_counts.items():
    pct = count / len(df) * 100
    print(f'  {label} : {count} ({pct:.1f}%)')

ratio = class_counts.max() / class_counts.min()
print(f'\nRatio majorité/minorité : {ratio:.2f}')
if ratio > 1.5:
    print('  Déséquilibre modéré détecté - à surveiller lors de la modélisation.')
else:
    print('  Classes quasi-équilibrées.')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Barplot
bars = axes[0].bar(class_counts.index, class_counts.values,
                   color=['#4C9BE8', '#E8634C'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Distribution des classes', fontweight='bold')
axes[0].set_ylabel('Nombre d\'observations')
axes[0].set_ylim(0, class_counts.max() * 1.3)
for bar, val in zip(bars, class_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val}\n({val/len(df)*100:.1f}%)', ha='center', va='bottom', fontsize=10)

# Pie chart
axes[1].pie(class_counts.values, labels=class_counts.index,
            colors=['#4C9BE8', '#E8634C'], autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Proportion des classes', fontweight='bold')

plt.suptitle('Équilibre des classes - Classification (1 vs 2)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### A.4 - Histogrammes de distribution

In [ ]:
n = len(features)
ncols = 3
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(15, nrows * 3.5))
axes = axes.flatten()

palette = {'1': '#4C9BE8', '2': '#E8634C'}

for i, feat in enumerate(features):
    ax = axes[i]
    for cls, color in zip([1, 2], ['#4C9BE8', '#E8634C']):
        subset = df[df[target] == cls][feat]
        ax.hist(subset, bins=18, alpha=0.65, color=color, edgecolor='white', linewidth=0.5,
                label=f'Classe {cls}')

    # Courbe de densité KDE
    for cls, color in zip([1, 2], ['#2070C0', '#C03020']):
        subset = df[df[target] == cls][feat]
        kde_x = np.linspace(subset.min(), subset.max(), 200)
        kde = stats.gaussian_kde(subset)
        ax2 = ax.twinx()
        ax2.plot(kde_x, kde(kde_x), color=color, linewidth=1.8, alpha=0.8)
        ax2.set_yticks([])

    ax.set_title(feat, fontweight='bold')
    ax.set_xlabel('Valeur')
    ax.set_ylabel('Fréquence')
    if i == 0:
        ax.legend(fontsize=9)

# Masquer axes inutilisés
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Histogrammes des variables par classe\n(bleu = Contrôle, rouge = Cancer)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### A.5 - Boxplots (détection visuelle des outliers)

In [ ]:
fig, axes = plt.subplots(nrows, ncols, figsize=(15, nrows * 3.5))
axes = axes.flatten()

for i, feat in enumerate(features):
    ax = axes[i]
    data_plot = [df[df[target] == cls][feat].values for cls in [1, 2]]

    bp = ax.boxplot(data_plot,
                    labels=['Contrôle (1)', 'Cancer (2)'],
                    patch_artist=True,
                    medianprops={'color': 'black', 'linewidth': 2},
                    whiskerprops={'linewidth': 1.5},
                    capprops={'linewidth': 1.5},
                    flierprops={'marker': 'o', 'markersize': 5, 'alpha': 0.5})

    colors = ['#4C9BE8', '#E8634C']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    ax.set_title(feat, fontweight='bold')
    ax.set_ylabel('Valeur')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Boxplots des variables par classe\n(points = valeurs aberrantes potentielles)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### A.6 - Heatmap de corrélation

In [ ]:
corr_matrix = df[features + [target]].corr()

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

#  Heatmap complète
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, vmin=-1, vmax=1,
            linewidths=0.5, ax=axes[0],
            annot_kws={'size': 9})
axes[0].set_title('Matrice de corrélation (triangle inférieur)', fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

#  Corrélations avec la cible
target_corr = corr_matrix[target].drop(target).sort_values(key=abs, ascending=True)
colors_bar = ['#E8634C' if v > 0 else '#4C9BE8' for v in target_corr]
bars = axes[1].barh(target_corr.index, target_corr.values,
                    color=colors_bar, edgecolor='white', linewidth=0.8)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Corrélation avec Classification\n(rouge = positive, bleu = négative)', fontweight='bold')
axes[1].set_xlabel('Coefficient de corrélation de Pearson')
for bar, val in zip(bars, target_corr.values):
    axes[1].text(val + 0.01 * np.sign(val), bar.get_y() + bar.get_height()/2,
                 f'{val:.2f}', va='center', ha='left' if val > 0 else 'right', fontsize=9)

plt.suptitle('Analyse des corrélations', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n Top corrélations avec la cible ')
print(target_corr.sort_values(key=abs, ascending=False).to_string())

### A.7 - Détection des valeurs aberrantes (méthode IQR)

In [ ]:
print('=== Détection des valeurs aberrantes - Méthode IQR (seuil : 1.5 × IQR) ===\n')

outlier_summary = []
outlier_masks = {}

for feat in features:
    Q1 = df[feat].quantile(0.25)
    Q3 = df[feat].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    mask = (df[feat] < lower) | (df[feat] > upper)
    outlier_masks[feat] = mask
    n_out = mask.sum()
    pct = n_out / len(df) * 100

    outlier_summary.append({
        'Variable': feat,
        'Q1': round(Q1, 3), 'Q3': round(Q3, 3), 'IQR': round(IQR, 3),
        'Borne inf.': round(lower, 3), 'Borne sup.': round(upper, 3),
        'N outliers': n_out, '% outliers': round(pct, 1)
    })

summary_df = pd.DataFrame(outlier_summary).set_index('Variable')
print(summary_df.to_string())

# Observations ayant au moins 1 outlier
any_outlier = pd.concat(outlier_masks, axis=1).any(axis=1)
print(f'\nObservations avec ≥ 1 valeur aberrante : {any_outlier.sum()} / {len(df)} ({any_outlier.mean()*100:.1f}%)')

In [ ]:
# Visualisation : nombre d'outliers par variable
out_counts = summary_df['N outliers'].sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 4))
colors_out = ['#E8634C' if v > 0 else '#B0C4DE' for v in out_counts]
bars = ax.barh(out_counts.index, out_counts.values, color=colors_out, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Nombre de valeurs aberrantes (IQR × 1.5)')
ax.set_title('Valeurs aberrantes par variable', fontweight='bold')
for bar, val in zip(bars, out_counts.values):
    if val > 0:
        ax.text(val + 0.05, bar.get_y() + bar.get_height()/2,
                str(val), va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Z-score method (seuil |z| > 3) - comparaison
print('=== Méthode Z-score (|z| > 3) ===\n')
z_scores = np.abs(stats.zscore(df[features]))
z_outliers = (z_scores > 3).sum(axis=0)
z_df = pd.DataFrame({'Variable': features, 'N outliers (z>3)': z_outliers}).set_index('Variable')
print(z_df[z_df['N outliers (z>3)'] > 0].to_string() or 'Aucun outlier détecté (z > 3).')

**Décision sur les outliers** : Les valeurs aberrantes identifiées correspondent à des valeurs biologiquement plausibles (notamment pour l'Insuline, la Leptine et le MCP-1, dont les distributions sont fortement asymétriques à droite). Ces observations ne seront **pas supprimées** car elles peuvent véhiculer une information discriminante. La standardisation (étape B) limitera leur impact sur les modèles sensibles à l'échelle.


## B - Prétraitement des Données

### B.1 - Encodage de la variable cible

In [ ]:
# Recode : 1 → 0 (contrôle), 2 → 1 (cancer) - convention standard pour la classification binaire
df['y'] = (df[target] - 1).astype(int)  # 0 = sain, 1 = cancer

X = df[features].values
y = df['y'].values

print(f'X shape : {X.shape}')
print(f'y shape : {y.shape}')
print(f'Classes → 0 (contrôle sain) : {(y==0).sum()} | 1 (cancer) : {(y==1).sum()}')

### B.2 - Découpage train/test stratifié (80/20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y          # préserve la proportion des classes
)

print(f'Taille train : {X_train.shape[0]} observations ({X_train.shape[0]/len(df)*100:.1f}%)')
print(f'Taille test  : {X_test.shape[0]} observations ({X_test.shape[0]/len(df)*100:.1f}%)')
print(f'\nDistribution train → 0: {(y_train==0).sum()} | 1: {(y_train==1).sum()}')
print(f'Distribution test  → 0: {(y_test==0).sum()}  | 1: {(y_test==1).sum()}')

# Vérification de la stratification
train_ratio = y_train.mean()
test_ratio  = y_test.mean()
full_ratio  = y.mean()
print(f'\nProportion classe 1 - Total: {full_ratio:.3f} | Train: {train_ratio:.3f} | Test: {test_ratio:.3f}')
print(' Stratification respectée.' if abs(train_ratio - test_ratio) < 0.05 else '  Vérifier la stratification.')

### B.3 - Standardisation avec StandardScaler

In [ ]:
# Le scaler est UNIQUEMENT ajusté sur le train pour éviter toute fuite d'information (data leakage)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform sur train
X_test_scaled  = scaler.transform(X_test)        # transform seulement sur test

print('Paramètres du scaler (moyennes apprises sur train) :')
for feat, mean, std in zip(features, scaler.mean_, scaler.scale_):
    print(f'  {feat:15s} → μ = {mean:8.3f} | σ = {std:8.3f}')

print(f'\nX_train_scaled shape : {X_train_scaled.shape}')
print(f'X_test_scaled  shape : {X_test_scaled.shape}')

# Vérification : moyenne ≈ 0, std ≈ 1 sur le train
train_means = X_train_scaled.mean(axis=0)
train_stds  = X_train_scaled.std(axis=0)
print(f'\nVérification train scalé :')
print(f'  Moyenne max |μ| : {np.abs(train_means).max():.2e}  (attendu ≈ 0)')
print(f'  Écart-type max  : {np.abs(train_stds - 1).max():.2e}  (attendu ≈ 1)')

In [ ]:
# Visualisation : distributions avant / après standardisation pour 3 variables représentatives
sample_feats = ['Glucose', 'Insulin', 'Leptin']
sample_idx   = [features.index(f) for f in sample_feats]

fig, axes = plt.subplots(2, len(sample_feats), figsize=(14, 6))

for col, (feat, idx) in enumerate(zip(sample_feats, sample_idx)):
    # Avant
    axes[0, col].hist(X_train[:, idx], bins=20, color='#4C9BE8', edgecolor='white', alpha=0.8)
    axes[0, col].set_title(f'{feat}\n(brut)', fontweight='bold')
    axes[0, col].set_ylabel('Fréquence' if col == 0 else '')

    # Après
    axes[1, col].hist(X_train_scaled[:, idx], bins=20, color='#50C87C', edgecolor='white', alpha=0.8)
    axes[1, col].set_title(f'{feat}\n(standardisé)', fontweight='bold')
    axes[1, col].set_ylabel('Fréquence' if col == 0 else '')
    axes[1, col].set_xlabel('z-score')

fig.suptitle('Standardisation - Distributions avant / après (sur train)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### B.4 - Récapitulatif du prétraitement

In [ ]:
print('=' * 60)
print('RÉCAPITULATIF DU PRÉTRAITEMENT')
print('=' * 60)
print(f'  Dataset complet          : {len(df)} observations × {len(features)} features')
print(f'  Valeurs manquantes       : 0 (confirmé UCI)')
print(f'  Variable cible encodée   : 0 = sain | 1 = cancer')
print(f'  Découpage train/test     : 80% / 20% (stratifié sur y)')
print(f'  Train                    : {X_train_scaled.shape[0]} observations')
print(f'  Test                     : {X_test_scaled.shape[0]} observations')
print(f'  Standardisation          : StandardScaler (μ=0, σ=1) - fit sur train uniquement')
print(f'  Outliers                 : conservés (valeurs biologiquement plausibles)')
print()
print('Variables disponibles pour la modélisation :')
for f in features:
    print(f'  - {f}')
print()
print('Objets exportés :')
print('  X_train_scaled, X_test_scaled  → features standardisées')
print('  y_train, y_test                → labels (0/1)')
print('  scaler                         → StandardScaler ajusté')
print('  features                       → liste des noms de colonnes')

## C - Modélisation

### C.1 - Imports complémentaires

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_auc_score
)
import warnings
warnings.filterwarnings("ignore")
print("Imports modélisation done")

### C.2 - Justification des modèles choisis

Trois modèles de classification binaire ont été sélectionnés, couvrant des approches complémentaires :

| Modèle | Type | Justification |
|--------|------|---------------|
| **Régression Logistique** | Paramétrique, linéaire | Modèle de référence pour la classification binaire. Simple, interprétable via ses coefficients, efficace sur des données standardisées. |
| **K-Nearest Neighbors (KNN)** | Non-paramétrique | Aucune hypothèse sur la distribution des données. Particulièrement pertinent après standardisation, qui garantit une distance euclidienne cohérente entre les features. |
| **Gaussian Naive Bayes** | Probabiliste, génératif | Modèle adapté aux features numériques continues : il suppose une distribution gaussienne au sein de chaque classe, hypothèse raisonnablement vérifiable sur ce dataset. |

**Fonctions de coût et méthodes d'optimisation :**

- **Régression Logistique** : minimise la log-vraisemblance négative (entropie croisée binaire) par descente de gradient :
$$J(w, b) = -\frac{1}{n}\sum_{i=1}^{n} \left[ y_i \log(\hat{p}_i) + (1 - y_i) \log(1 - \hat{p}_i) \right]$$
avec $\hat{p}_i = \sigma(w^\top x_i + b)$, $\sigma$ étant la fonction sigmoïde. Le paramètre de régularisation $C = 1.0$ contrôle le compromis biais-variance.
- **KNN** : pas de phase d'apprentissage paramétrique. La prédiction repose sur le vote majoritaire des $k$ voisins les plus proches (distance euclidienne). La valeur optimale de $k$ est déterminée par validation croisée stratifiée (section C.4).
- **Gaussian Naive Bayes** : les paramètres $\mu_{c,j}$ et $\sigma^2_{c,j}$ (moyenne et variance de chaque feature $j$ dans chaque classe $c$) sont estimés par maximum de vraisemblance directement sur le jeu d'entraînement. Aucune optimisation itérative n'est requise.

### C.3 - Modèle 1 : Régression Logistique

In [ ]:
# Entraînement
lr = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, C=1.0)
lr.fit(X_train_scaled, y_train)

# Prédictions
y_pred_lr = lr.predict(X_test_scaled)
y_proba_lr = lr.predict_proba(X_test_scaled)[:, 1]

print("=== Régression Logistique ===")
print(f"Accuracy  : {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"AUC-ROC   : {roc_auc_score(y_test, y_proba_lr):.4f}")
print()
print("Rapport de classification :")
print(classification_report(y_test, y_pred_lr, target_names=["Contrôle (0)", "Cancer (1)"]))

# Coefficients du modèle
print("Coefficients du modèle (impact de chaque biomarqueur) :")
coef_df = pd.DataFrame({
    "Feature": features,
    "Coefficient": lr.coef_[0].round(4)
}).sort_values("Coefficient", key=abs, ascending=False)
print(coef_df.to_string(index=False))

### C.4 - Modèle 2 : K-Nearest Neighbors (KNN) — Choix de k optimal

In [ ]:
# Recherche du k optimal par cross-validation (5-fold) sur le train
k_values = range(1, 16)
cv_scores = []

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for k in k_values:
    knn_cv = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn_cv, X_train_scaled, y_train, cv=cv, scoring="accuracy")
    cv_scores.append(scores.mean())

best_k = k_values[np.argmax(cv_scores)]
print(f"Meilleur k trouvé par cross-validation : k = {best_k}  (CV accuracy = {max(cv_scores):.4f})")

# Visualisation
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(k_values, cv_scores, marker="o", color="#4C9BE8", linewidth=2, markersize=6)
ax.axvline(best_k, color="#E8634C", linestyle="--", linewidth=1.5, label=f"k optimal = {best_k}")
ax.set_xlabel("Nombre de voisins k")
ax.set_ylabel("Accuracy (CV 5-fold)")
ax.set_title("Choix de k optimal par cross-validation", fontweight="bold")
ax.legend()
ax.set_xticks(list(k_values))
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Entraînement avec le k optimal
knn = KNeighborsClassifier(n_neighbors=best_k)
knn.fit(X_train_scaled, y_train)

y_pred_knn = knn.predict(X_test_scaled)
y_proba_knn = knn.predict_proba(X_test_scaled)[:, 1]

print(f"=== K-Nearest Neighbors (k={best_k}) ===")
print(f"Accuracy  : {accuracy_score(y_test, y_pred_knn):.4f}")
print(f"AUC-ROC   : {roc_auc_score(y_test, y_proba_knn):.4f}")
print()
print("Rapport de classification :")
print(classification_report(y_test, y_pred_knn, target_names=["Contrôle (0)", "Cancer (1)"]))

### C.5 - Modèle 3 : Gaussian Naive Bayes

In [ ]:
# GaussianNB suppose que chaque feature suit une loi normale au sein de chaque classe
# Les paramètres (moyenne et variance par classe) sont estimés par maximum de vraisemblance
gnb = GaussianNB()
gnb.fit(X_train_scaled, y_train)

y_pred_gnb = gnb.predict(X_test_scaled)
y_proba_gnb = gnb.predict_proba(X_test_scaled)[:, 1]

print("=== Gaussian Naive Bayes ===")
print(f"Accuracy  : {accuracy_score(y_test, y_pred_gnb):.4f}")
print(f"AUC-ROC   : {roc_auc_score(y_test, y_proba_gnb):.4f}")
print()
print("Rapport de classification :")
print(classification_report(y_test, y_pred_gnb, target_names=["Contrôle (0)", "Cancer (1)"]))

# Paramètres gaussiens appris par le modèle
print("Paramètres appris (moyennes par classe) :")
for cls_idx, cls_name in enumerate(["Contrôle (0)", "Cancer (1)"]):
    print(f"  Classe {cls_name}:")
    for feat, mean_val, var_val in zip(features, gnb.theta_[cls_idx], gnb.var_[cls_idx]):
        print(f"    {feat:15s} → μ = {mean_val:7.3f} | σ² = {var_val:.3f}")

## D - Évaluation des Performances

### D.1 - Justification des métriques

Dans un contexte médical de dépistage du cancer du sein, toutes les métriques n'ont pas le même poids clinique :

| Métrique | Définition | Rôle dans ce projet |
|----------|------------|---------------------|
| **Recall** (sensibilité) | $\frac{TP}{TP + FN}$ | **Métrique prioritaire** : mesure la proportion de cas cancéreux effectivement détectés. Un faux négatif (cancer non détecté) est cliniquement bien plus grave qu'un faux positif. |
| **Precision** | $\frac{TP}{TP + FP}$ | Proportion de prédictions positives réellement correctes. Un faux positif génère de l'anxiété et des examens inutiles, mais reste moins grave. |
| **F1-score** | $\frac{2 \times \text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$ | Moyenne harmonique — indicateur global équilibré entre précision et sensibilité. |
| **Accuracy** | $\frac{TP + TN}{n}$ | Taux global de bonnes classifications. Moins informative en présence d'un déséquilibre de classes, présentée à titre indicatif. |

> **Critère principal d'évaluation** : le **Recall sur la classe 1 (cancer)** est la métrique déterminante pour comparer les modèles, en cohérence avec l'objectif médical de minimisation des faux négatifs.

### D.2 - Matrices de confusion

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

model_results = [
    ("Régression Logistique", y_pred_lr,  "#4C9BE8"),
    (f"KNN (k={best_k})",      y_pred_knn, "#E8634C"),
    ("Gaussian Naive Bayes",  y_pred_gnb, "#50C87C"),
]

for ax, (name, y_pred, color) in zip(axes, model_results):
    cm = confusion_matrix(y_test, y_pred)

    # Affichage heatmap
    im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    ax.set_title(name, fontweight="bold", fontsize=11)

    tick_marks = [0, 1]
    ax.set_xticks(tick_marks)
    ax.set_yticks(tick_marks)
    ax.set_xticklabels(["Prédit : Sain", "Prédit : Cancer"], rotation=20, ha="right")
    ax.set_yticklabels(["Réel : Sain", "Réel : Cancer"])

    # Annotations avec couleur contrastée
    thresh = cm.max() / 2.0
    for i in range(2):
        for j in range(2):
            label = "TN" if (i==0 and j==0) else "FP" if (i==0 and j==1) else "FN" if (i==1 and j==0) else "TP"
            ax.text(j, i, f"{cm[i,j]} ({label})",
                    ha="center", va="center", fontsize=12, fontweight="bold",
                    color="white" if cm[i,j] > thresh else "black")

plt.suptitle("Matrices de confusion — comparaison des 3 modèles", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("Légende : TN = Vrais Négatifs | FP = Faux Positifs | FN = Faux Négatifs | TP = Vrais Positifs")
print("         FN = cas cancer NON détectés → à minimiser en priorité")

In [ ]:
# Construction du tableau récapitulatif
results = []
for name, y_pred in [
    ("Régression Logistique", y_pred_lr),
    (f"KNN (k={best_k})",      y_pred_knn),
    ("Gaussian Naive Bayes",  y_pred_gnb),
]:
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    results.append({
        "Modèle":        name,
        "Accuracy":      round(float(accuracy_score(y_test, y_pred)), 4),
        "Precision":     round(float(precision_score(y_test, y_pred)), 4),
        "Recall":        round(float(recall_score(y_test, y_pred)), 4),
        "F1-score":      round(float(f1_score(y_test, y_pred)), 4),
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn)
    })

results_df = pd.DataFrame(results).set_index("Modèle")
print("=" * 70)
print("TABLEAU COMPARATIF DES PERFORMANCES")
print("=" * 70)
print(results_df.to_string())
print()

# Meilleur modèle selon chaque métrique
for metric in ["Accuracy", "Recall", "F1-score"]:
    best = results_df[metric].idxmax()
    print(f"  Meilleur {metric:10s} : {best}  ({results_df.loc[best, metric]:.4f})")

### D.3 - Visualisation des métriques


In [ ]:
metrics_plot = ["Accuracy", "Precision", "Recall", "F1-score"]
x = np.arange(len(metrics_plot))
width = 0.25
colors_bar = ["#4C9BE8", "#E8634C", "#50C87C"]
model_names = results_df.index.tolist()

fig, ax = plt.subplots(figsize=(13, 5))

for i, (model_name, color) in enumerate(zip(model_names, colors_bar)):
    vals = [results_df.loc[model_name, m] for m in metrics_plot]
    bars = ax.bar(x + i * width, vals, width, label=model_name,
                  color=color, edgecolor="white", linewidth=0.8, alpha=0.9)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{val:.2f}", ha="center", va="bottom", fontsize=8, fontweight="bold")

ax.set_xticks(x + width)
ax.set_xticklabels(metrics_plot, fontsize=11)
ax.set_ylabel("Score")
ax.set_ylim(0, 1.15)
ax.set_title("Comparaison des métriques — 3 modèles", fontweight="bold", fontsize=13)
ax.legend(loc="upper right", fontsize=9)
ax.axhline(0.8, color="gray", linestyle=":", linewidth=1, alpha=0.6)
ax.text(len(metrics_plot) - 0.1, 0.81, "seuil 0.80", color="gray", fontsize=8)
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## E - Comparaison des modèles et discussion

Cette partie permet de comparer les trois modèles de manière plus approfondie, en tenant compte non seulement des scores numériques, mais aussi de la nature des erreurs commises. Dans un problème médical de dépistage, l'objectif principal est de limiter les **faux négatifs**, c'est-à-dire les patientes malades classées comme non malades.

### E.1 - Discussion et choix du modèle final

Le tableau ci-dessous synthétise les performances des trois modèles sur le jeu de test (24 observations) :

| Modèle | Accuracy | Precision | Recall | F1-score |
|--------|----------|-----------|--------|----------|
| Régression Logistique | 0.7917 | 0.83 | 0.77 | 0.80 |
| **KNN (k optimal)** | **0.8333** | **0.87** | **0.77** | **0.83** |
| Gaussian Naive Bayes | 0.6667 | 0.77 | 0.46 | 0.60 |

**Analyse des résultats :**

- Le **KNN** présente les meilleures performances globales en termes d'Accuracy (83.3%) et de F1-score (0.83). Il constitue le meilleur compromis entre sensibilité et spécificité sur ce dataset.
- La **Régression Logistique** offre des performances solides et stables (Accuracy 79.2%, F1 0.80), avec l'avantage d'une interprétabilité directe via les coefficients appris.
- Le **Gaussian Naive Bayes** affiche un Recall de seulement 0.46 sur la classe cancer — plus de la moitié des cas cancéreux ne sont pas détectés — ce qui est incompatible avec une application médicale.

**Modèle retenu :** Le KNN est sélectionné comme modèle final en raison de ses meilleures performances globales. La Régression Logistique est conservée comme modèle de référence interprétable.

**Limites :** Le jeu de test ne contient que 24 observations, ce qui confère une forte variabilité aux estimations de performance. Les résultats doivent être interprétés avec prudence ; une validation sur un jeu de données plus large serait nécessaire pour confirmer ces conclusions.

### E.2 - Comparaison globale des performances

D'après les métriques obtenues précédemment, le modèle **KNN** fournit les meilleures performances globales. Il obtient la meilleure Accuracy et le meilleur F1-score. La **Régression Logistique** reste très proche du KNN, avec l'avantage d'être plus facile à interpréter. Le modèle **Gaussian Naive Bayes** est le moins performant, surtout au niveau du Recall.

Le Recall est particulièrement important dans ce projet, car il mesure la capacité du modèle à détecter correctement les patientes atteintes d'un cancer. Un Recall faible signifie qu'un nombre important de patientes malades ne sont pas détectées.

In [ ]:
# Comparaison synthétique des modèles
comparison_df = results_df.copy()
comparison_df["Ranking_F1"] = comparison_df["F1-score"].rank(ascending=False, method="min").astype(int)
comparison_df["Ranking_Recall"] = comparison_df["Recall"].rank(ascending=False, method="min").astype(int)
comparison_df.sort_values(by=["F1-score", "Recall"], ascending=False)

### E.3 - Analyse des erreurs de classification

Les matrices de confusion permettent d'identifier les types d'erreurs réalisées par chaque modèle :

- **Faux positif (FP)** : une patiente saine est prédite comme malade ;
- **Faux négatif (FN)** : une patiente malade est prédite comme saine.

Dans ce contexte, les **faux négatifs** sont les erreurs les plus graves, car ils correspondent à des cas de cancer non détectés. Le modèle le plus intéressant est donc celui qui garde un bon équilibre entre Accuracy, Precision, Recall et F1-score, tout en réduisant au maximum le nombre de faux négatifs.

In [ ]:
# Analyse détaillée des erreurs à partir des matrices de confusion
error_analysis = []

for name, y_pred in [
    ("Régression Logistique", y_pred_lr),
    (f"KNN (k={best_k})", y_pred_knn),
    ("Gaussian Naive Bayes", y_pred_gnb),
]:
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    error_analysis.append({
        "Modèle": name,
        "Vrais négatifs (TN)": tn,
        "Faux positifs (FP)": fp,
        "Faux négatifs (FN)": fn,
        "Vrais positifs (TP)": tp,
        "Erreur critique (FN)": fn
    })

error_df = pd.DataFrame(error_analysis).set_index("Modèle")
error_df

### E.4 - Discussion du choix du modèle

Le choix final du modèle dépend de l'objectif principal :

- Si l'objectif est la **meilleure performance globale**, le **KNN** est le meilleur choix, car il possède les meilleurs scores en Accuracy et en F1-score.
- Si l'objectif est l'**interprétabilité**, la **Régression Logistique** est une très bonne alternative. Elle permet d'expliquer l'effet des variables biologiques sur la probabilité d'appartenir à la classe cancer.
- Le **Gaussian Naive Bayes** n'est pas retenu, car son Recall est trop faible pour un problème médical.

Ainsi, le **KNN est retenu comme modèle final** dans ce projet. Cependant, la Régression Logistique reste un modèle de référence intéressant, surtout dans un contexte médical où l'explicabilité des décisions est importante.

### E.5 - Limites de la comparaison

La comparaison des modèles doit être interprétée avec prudence pour plusieurs raisons :

- Le dataset contient seulement **116 observations**, ce qui est faible pour entraîner et évaluer des modèles de machine learning.
- Le jeu de test contient environ **24 observations**, donc une seule erreur peut fortement modifier les métriques.
- Les modèles ont été évalués sur un seul découpage train/test ; une validation croisée donnerait une estimation plus robuste.
- Certaines variables présentent des valeurs extrêmes, ce qui peut influencer les modèles sensibles aux distances comme le KNN.

Ces limites montrent que les résultats obtenus sont surtout exploratoires.

## F - Conclusion

L'objectif de ce projet était de construire un modèle de classification capable de prédire la présence d'un cancer du sein à partir de variables biologiques issues du dataset **Breast Cancer Coimbra**.

Le travail a commencé par une analyse exploratoire des données afin d'étudier les distributions des variables, les corrélations, les valeurs extrêmes et l'équilibre des classes. Ensuite, les données ont été standardisées avec **StandardScaler**, puis divisées en un jeu d'entraînement et un jeu de test selon une répartition 80% / 20% stratifiée sur la variable cible.

Trois modèles de classification ont été comparés :

- la **Régression Logistique** ;
- le **K-Nearest Neighbors (KNN)** ;
- le **Gaussian Naive Bayes**.

Les résultats montrent que le modèle **KNN** obtient les meilleures performances globales, avec la meilleure Accuracy et le meilleur F1-score. La Régression Logistique donne également de bons résultats et reste intéressante grâce à son interprétabilité. En revanche, Gaussian Naive Bayes est moins adapté à ce problème, car son Recall est trop faible.

En conclusion, le **KNN** est retenu comme modèle final pour cette étude. Toutefois, les résultats doivent être considérés avec prudence, car le dataset est de petite taille. Pour améliorer la fiabilité du modèle, il serait nécessaire de tester les performances sur un échantillon plus grand et d'utiliser une validation croisée.

### F.1 - Perspectives d'amélioration

Plusieurs améliorations peuvent être proposées pour prolonger ce travail :

1. Utiliser une **validation croisée** afin d'obtenir des résultats plus stables.
2. Tester d'autres modèles comme les **arbres de décision**, les **Random Forest**, les **SVM** ou le **Gradient Boosting**.
3. Étudier l'importance des variables pour identifier les biomarqueurs les plus influents.
4. Traiter plus finement les valeurs extrêmes afin de limiter leur influence.
5. Utiliser un dataset plus volumineux pour améliorer la généralisation des modèles.
6. Ajuster les seuils de classification pour augmenter le Recall et réduire les faux négatifs.

Ces perspectives permettraient d'obtenir un modèle plus robuste et plus adapté à une utilisation dans un contexte médical réel.